In [1]:
import argparse
from datetime import datetime
import logging
import sys
import time
from time import perf_counter_ns
import traceback
from queue import Queue, Empty
from signal import SIGINT, signal, strsignal
from threading import Thread, Event
from types import FrameType

import nidaqmx
import json
import nidaqmx.system
from concurrent.futures import ThreadPoolExecutor, as_completed, TimeoutError
import queue
import os
from device.Heatingstage import heat_stage_group

In [3]:
import nidaqmx
print(nidaqmx.__version__)

0.9.0


In [2]:
class heat_stage_group():
    def __init__(self,config_file, temp,pos_path):
        self.heat_stage_id_ls=config_file
        self.heat_mode=False
        self.image_mode=True
        self.ao_port="Dev3/ao0"    
        self.temp=temp
        self.cancel=0
        self.num_devices=len(self.heat_stage_id_ls)
        self.pos_path=pos_path
        self.cancel=0
    def connect_heater_group(self):
        self.do_task=[None]*self.num_devices
        self.ai_task=[None]*self.num_devices
        for i in range(self.num_devices):
            self.do_task[i]=nidaqmx.Task()
            self.ai_task[i]=nidaqmx.Task()
            do_channel = self.heat_stage_id_ls[i]['DAQ']+'/'+self.heat_stage_id_ls[i]['do_port']+'/'+self.heat_stage_id_ls[i]['do_line']
            ai_channel=self.heat_stage_id_ls[i]['DAQ']+'/'+self.heat_stage_id_ls[i]['ai_line']
            self.do_task[i].do_channels.add_do_chan(do_channel)
            self.ai_task[i].ai_channels.add_ai_voltage_chan(ai_channel)
            self.do_task[i].start()
            self.do_task[i].write(self.image_mode)
            self.ai_task[i].start()
        self.ao_task=nidaqmx.Task()
        self.ao_task.ao_channels.add_ao_voltage_chan(self.ao_port, min_val=0.0, max_val=5.0)
        self.ao_task.start()
    def disconnect_heater_group(self):
        self.ao_task.stop()
        self.ao_task.close()
        for i in range(self.num_devices):
            self.ai_task[i].stop()
            self.ai_task[i].close()
            self.do_task[i].stop()
            self.do_task[i].close()
    def start_heat(self):
        self.ao_task.write(self.temp)
        for i in range(self.num_devices):
                self.do_task[i].write(self.heat_mode)
        print("start heat")
    def end_heat(self):
        self.ao_task.write(0.1)
        print("end heat")
    def heat_for_3min_single(self,i):
         try:
            temp=self.ai_task[i].read()
            start_single = datetime.now()
            time_diff_single=0
            while temp<=self.temp-0.3 and time_diff_single<=180 and self.cancel==0:
                    time.sleep(5)
                    time_diff=(datetime.now()-start_single).total_seconds()
                    temp = self.ai_task[i].read()
                    print(f"device {i} temp: {temp}")
            if time_diff_single>=180 and temp<=self.temp-0.5:
                    print(f"Heating stage had an issue: Device {i} is off!")
                    result =0
            else:
                    print(f"start stable heat for 3 mins for Device {i}")
                    start_heat = datetime.now()
                    heat_time_diff = 0
                    while heat_time_diff<=180 and self.cancel==0 :
                        temp = self.ai_task[i].read()
                        print("device"+str(i)+" temp: "+str(temp))
                        heat_time_diff=(datetime.now()-start_heat).total_seconds()
                        time.sleep(10)
                    self.do_task[i].write(self.image_mode)
                    result =1
            self.result_queue.put((i, result))        
         except Exception as e:
            self.result_queue.put((i, f"Error in device {i}: {str(e)}"))       
    
    def heat_for_3min(self):
        self.start_heat()
        start = datetime.now()
        time_diff=0
        self.result_queue = queue.Queue()
        with ThreadPoolExecutor(max_workers=self.num_devices) as executor:
            future_to_device = {executor.submit(self.heat_for_3min_single, i): i for i in range(self.num_devices)}
            try:
                for future in as_completed(future_to_device.keys(), timeout=10 * 60):
                    device_id = future_to_device[future]
                    try:
                        future.result()  # This will raise an exception if the thread raised one
                    except Exception as e:
                        print(f"Thread for device {device_id} raised an exception: {str(e)}")
            except TimeoutError:
                print(f"Timeout occurred after 10 minutes")
        results = []
        while not self.result_queue.empty():
            results.append(self.result_queue.get())
        self.end_heat()
        return results
    def check_temp(self):  
        for i in range(self.num_devices):
            value=self.ai_task[i].read()
            print(value)
    
    def write_log(self,txt):
        f = open(os.path.join(self.pos_path,"log.txt"), "a")
        f.write(txt)
        f.close()

In [2]:
with open(os.path.join("config_file","Heat_stage.json")) as f:
    cfg = json.load(f)
heat_stage_id_ls=cfg 

In [3]:
cfg[0]["DAQ"]+"/"+cfg[0]["ao_port"]

'Dev4/ao0'

In [4]:
num_devices=len(cfg)

In [8]:
heater=heat_stage_group(cfg, 6,"C://")

In [5]:
ai_task = nidaqmx.Task()
ai_channel = cfg[0]['DAQ'] + '/' + cfg[0]['ai_line']
ai_task.ai_channels.add_ai_voltage_chan(ai_channel)

AIChannel(name=Dev4/ai0)

In [6]:
temp = ai_task.read()
temp

-0.17988513503223658

In [8]:
ai_task.close()

In [ ]:
ao_task.ao_channels.add_ao_voltage_chan("ai0", min_val=0.0, max_val=5.0)

In [ ]:
heater.check_temp()

In [ ]:
heater.disconnect_heater_group()

In [ ]:
### during heat period
heater.heat_for_3min()

In [ ]:
heater.end_heat()

In [ ]:

heater.disconnect_heater_group()

In [ ]:
with open("Heat_stage.json") as f:
    cfg = json.load(f)

In [ ]:
task = nidaqmx.Task()


In [ ]:
import nidaqmx
import nidaqmx.system

system = nidaqmx.system.System.local()

# Get all connected devices
connected_devices = system.devices

for device in connected_devices:
    print(f"Connected Device: {device.name}")
    
    
    # Check for analog output ports
    ao_ports = device.ao_physical_chans
    if ao_ports:
        print("  Analog Output Ports:")
        for port in ao_ports:
            print(f"    {port.name}")
    
    # Check for digital input/output ports
    print("\n")

if not connected_devices:
    print("No devices detected. Please check your connections.")

In [ ]:
## image mode dev3
task = nidaqmx.Task()
task.do_channels.add_do_chan("Dev3/port1/line2") 
task.start()
value = True
task.write(value)
task.stop
task.close()

In [ ]:
## image mode dev2
task = nidaqmx.Task()
task.do_channels.add_do_chan("Dev3/port1/line1") 
task.start()
value = True
task.write(value)
task.stop
task.close()

In [ ]:
## heat mode
task = nidaqmx.Task()
task.do_channels.add_do_chan("Dev3/port1/line1") 
task.start()
value = False
task.write(value)
task.stop
task.close()

In [ ]:
type(value)

In [ ]:
task.stop
task.close()

In [ ]:
task = nidaqmx.Task()
task.ai_channels.add_ai_voltage_chan("Dev3/ai0")
task.ai_channels.add_ai_voltage_chan("Dev3/ai1")
task.ai_channels.add_ai_voltage_chan("Dev3/ai2")
task.start()
value = task.read()[2]
print(value)
task.stop
task.close()

## read temp

In [35]:
task = nidaqmx.Task()
task.ai_channels.add_ai_voltage_chan("Dev4/ai0")
task.start()
value = task.read()
print(value)
task.stop
task.close()

2.8062237625708804


In [36]:
task = nidaqmx.Task()
task.ai_channels.add_ai_voltage_chan("Dev4/ai1")
task.start()
value = task.read()
print(value)
task.stop
task.close()

2.9300537643721327


## set temp

In [30]:
import nidaqmx.system
device_name="Dev4"
output_channel="ao0"
task = nidaqmx.Task()
task.ao_channels.add_ao_voltage_chan(f'{device_name}/{output_channel}', min_val=0.0, max_val=5.0)
task.start()
task.write(0.1)
task.stop
task.close()

## set temp mode or image mode

In [28]:
image_mode=True
heat_mode=False
do_task_dev1 = nidaqmx.Task()
do_task_dev2 = nidaqmx.Task()
do_task_dev3 = nidaqmx.Task()
do_task_dev1.do_channels.add_do_chan("Dev4/port1/line0") 
do_task_dev2.do_channels.add_do_chan("Dev4/port1/line1") 
do_task_dev3.do_channels.add_do_chan("Dev4/port1/line2") 

DOChannel(name=Dev4/port1/line2)

In [29]:
image_mode

True

In [30]:
do_task_dev1.write(image_mode)
do_task_dev2.write(image_mode)
do_task_dev3.write(image_mode)

1

In [ ]:

do_task_dev2.write(False)


In [ ]:
do_task_dev1.stop()
do_task_dev2.stop()
do_task_dev3.stop()

In [ ]:
do_task_dev1.close()
do_task_dev2.close()
do_task_dev3.close()

In [ ]:
config_file

In [ ]:
heat_stage_id_ls=config_file
cold_mode=[True]*len(heat_stage_id_ls)
heat_mode=[False]*len(heat_stage_id_ls)
ao_port="Dev1/ao0"    
heat_limit=heat_limit

In [ ]:
ai_task = nidaqmx.Task()
ai_task.ai_channels.add_ai_voltage_chan("Dev3/ai0")
ai_task.start()
value = ai_task.read()
print(value)
ai_task.stop
ai_task.close()

In [ ]:
ai_task = nidaqmx.Task()
ai_task.ai_channels.add_ai_voltage_chan("Dev3/ai1")
ai_task.start()
value = ai_task.read()
print(value)
ai_task.stop
ai_task.close()

In [3]:
ai_task = nidaqmx.Task()
ai_task.ai_channels.add_ai_voltage_chan("Dev3/ai3")
ai_task.start()
value = ai_task.read()
print(value)
ai_task.stop
ai_task.close()

-0.15121078211814165


In [ ]:
import nidaqmx
import nidaqmx.system

system = nidaqmx.system.System.local()

# Get all connected devices
connected_devices = system.devices

for device in connected_devices:
    print(f"Connected Device: {device.name}")
    
    # Check for analog input ports
    ai_ports = device.ai_physical_chans
    if ai_ports:
        print("  Analog Input Ports:")
        for port in ai_ports:
            print(f"    {port.name}")
    
    # Check for analog output ports
    ao_ports = device.ao_physical_chans
    if ao_ports:
        print("  Analog Output Ports:")
        for port in ao_ports:
            print(f"    {port.name}")
    
    # Check for digital input/output ports
    dio_ports = device.do_ports
    if dio_ports:
        print("  Digital I/O Ports:")
        for port in dio_ports:
            print(f"    {port.name}")
    
    print("\n")

if not connected_devices:
    print("No devices detected. Please check your connections.")

In [13]:
from device.Heatingstage import heat_stage_group
import os
import json
import nidaqmx

In [14]:
with open(os.path.join("config_file", "Heat_stage.json"), 'r') as r:
    Heater_cfg = json.load(r)
pos_path=os.path.join("D://","test")
pos_path
num_devices=len(Heater_cfg)
heat_stage_id_ls=Heater_cfg
image_mode=True
heat_mode = False

In [15]:
heat_stage_id_ls

[{'heat_stage': 'heatstage1',
  'do_port': 'port1',
  'do_line': 'line0',
  'ai_line': 'ai0',
  'DAQ': 'Dev4',
  'ao_port': 'ao0'},
 {'heat_stage': 'heatstage2',
  'do_port': 'port1',
  'do_line': 'line1',
  'ai_line': 'ai1',
  'DAQ': 'Dev4',
  'ao_port': 'ao0'},
 {'heat_stage': 'heatstage3',
  'do_port': 'port1',
  'do_line': 'line2',
  'ai_line': 'ai3',
  'DAQ': 'Dev4',
  'ao_port': 'ao0'}]

In [16]:
ai_task = [None] * num_devices
do_task = [None] * num_devices
ai_task[0]=nidaqmx.Task()
ai_channel = heat_stage_id_ls[0]['DAQ'] + '/' + heat_stage_id_ls[0]['ai_line']
ai_task[0].ai_channels.add_ai_voltage_chan(ai_channel)
ai_task[1]=nidaqmx.Task()
ai_channel = heat_stage_id_ls[1]['DAQ'] + '/' + heat_stage_id_ls[1]['ai_line']
ai_task[1].ai_channels.add_ai_voltage_chan(ai_channel)
ai_task[2]=nidaqmx.Task()
ai_channel = heat_stage_id_ls[2]['DAQ'] + '/' + heat_stage_id_ls[2]['ai_line']
ai_task[2].ai_channels.add_ai_voltage_chan(ai_channel)
do_task[0]=nidaqmx.Task()
do_channel = heat_stage_id_ls[0]['DAQ'] + '/' + heat_stage_id_ls[0]['do_port'] + '/'+heat_stage_id_ls[0]['do_line']
do_task[0].do_channels.add_do_chan(do_channel)
do_task[1]=nidaqmx.Task()
do_channel = heat_stage_id_ls[1]['DAQ'] + '/' + heat_stage_id_ls[1]['do_port'] + '/'+heat_stage_id_ls[1]['do_line']
do_task[1].do_channels.add_do_chan(do_channel)
do_task[2]=nidaqmx.Task()
do_channel = heat_stage_id_ls[2]['DAQ'] + '/' + heat_stage_id_ls[2]['do_port'] + '/'+heat_stage_id_ls[2]['do_line']
do_task[2].do_channels.add_do_chan(do_channel)

DaqError: The specified device is not present or is not active in the system. The device may not be installed on this system, may have been unplugged, or may not be installed correctly.

Device:  Dev4

Task Name: _unnamedTask<2>

Status Code: -88705

In [9]:
ao_port = "Dev3/ao0"
ao_task = nidaqmx.Task()
ao_task.ao_channels.add_ao_voltage_chan(ao_port, min_val=0.0, max_val=5.0)
ao_task.write(0.03)

DaqWriteError: The specified resource is reserved. The operation could not be completed as specified.

Task Name: _unnamedTask<3>

Status Code: -50103

In [7]:
ai_task[1].read()

NameError: name 'ai_task' is not defined

In [8]:
ai_task[0].read()

3.688746060244739

In [9]:
ai_task[2].read()

5.078407861059532

In [10]:
do_task[0].write(heat_mode)
do_task[1].write(heat_mode)
do_task[2].write(heat_mode)

1

In [5]:
do_task[0].write(image_mode)
do_task[1].write(image_mode)
do_task[2].write(image_mode)

1

In [12]:
ao_task.close()
ai_task[0].close()
ai_task[1].close()
ai_task[2].close()
do_task[0].close()
do_task[1].close()
do_task[2].close()

In [3]:
heatingdevice=heat_stage_group(Heater_cfg,5,pos_path)

In [4]:
heatingdevice.connect_heater_group()

0
1
2


In [ ]:
heatingdevice.end_heat()

In [ ]:
heatingdevice.disconnect_heater_group()

In [5]:
Heatingdevice.heat_for_3min()
print("Done")

start heat
start stable heat for 3 mins for Device 1
device1 temp: 4.786798235261813
device 0 temp: 3.935194903286174
device 0 temp: 3.9997103072237223
device 2 temp: 4.0719675596337765
device1 temp: 4.626800033496693
device 0 temp: 3.9055178174749017
device 2 temp: 4.065516019240022
device 0 temp: 3.7532614641822875
device 2 temp: 4.133902347413823
device1 temp: 4.746798684820533
device 0 temp: 3.9429367517586797
device 2 temp: 4.252610690658912
device 0 temp: 3.7984222469385713
device 2 temp: 4.41002827626653
device1 temp: 5.047440467169508
device 0 temp: 3.928743362892419
device 2 temp: 4.5971229476854205
device 0 temp: 3.931323979049921
device 2 temp: 4.790669159498066
start stable heat for 3 mins for Device 2
device2 temp: 4.781637002946809
device1 temp: 5.008731224806979
device 0 temp: 4.042290473822504
device 0 temp: 4.028097084956244
device2 temp: 4.97002198244445
device1 temp: 5.004860300570726
device 0 temp: 4.14293450396508
device 0 temp: 4.100354337366298
device2 temp: 4.92

In [4]:
import os
import json
from device.Heatingstage import heat_stage_group

In [5]:
with open(os.path.join("config_file", "Heat_stage.json"), 'r') as r:
    Heater_cfg = json.load(r)
pos_path=os.path.join("D://","test")

In [7]:
Heatingdevice=heat_stage_group(Heater_cfg,4,pos_path)



In [8]:
Heatingdevice.connect_heater_group()

C:\Users\barscope2\anaconda3\envs\seq_o_matic\lib\site-packages\nidaqmx\task.py:98: ResourceWarning: Task of name "_unnamedTask<0>" was not explicitly closed before it was destructed. Resources on the task device may still be reserved.
  warnings.warn(


DaqError: The specified device is not present or is not active in the system. The device may not be installed on this system, may have been unplugged, or may not be installed correctly.

Device:  Dev4

Task Name: _unnamedTask<1>

Status Code: -88705

In [6]:
Heatingdevice.check_temp()

-5.5188923897221684


In [14]:
Heatingdevice.heat_for_3min(60)

start heat
start stable heat 60 mins for Device heatstage1
heat for 60
device heatstage1  temp: 6.184201884549111 
device heatstage1  temp: 6.206137121887878 
device heatstage1  temp: 6.095170627115294 
device heatstage1  temp: 6.256459136959165 
device heatstage1  temp: 6.065493541304022 
device heatstage1  temp: 6.164847263367847 
device heatstage1  temp: 6.079686930170283 
End heat
